# 制定并执行清洗规则

In [4]:
# 导包
import pandas as pd
from config.config import REPORT_DIR,RAW_DATA_FILE
from src.data_loader import load_data

### 读取数据质量分析报告


In [2]:
report_path = REPORT_DIR/'data_quality_report.csv'
quality_report = load_data(report_path)
print(quality_report)

csv数据读取成功！
数据规模：6行 4列
   检查项目                    检查方法               问题描述             处理建议
0  字段缺失                isnull()          部分字段存在缺失值  根据字段业务含义决定删除或填充
1  重复记录            duplicated()     发现 34335 条重复记录   确认重复原因后再决定是否删除
2  负数数量            Quantity < 0    发现 22950 条负数量记录     结合取消订单业务规则判断
3   零数量           Quantity == 0        发现 0 条零数量记录       检查是否属于无效交易
4  异常单价  Price < 0 或 Price == 0    发现 6207 条非正单价记录     检查赠品、退款或异常交易
5  取消订单          Invoice 以 C 开头  发现 19494 条 C 开头记录  验证本地数据是否适用该识别规则


# 数据清洗计划（Data Cleaning Plan）

## 一、清洗目标

本阶段主要对 Online Retail II 原始数据集进行清洗和预处理，解决数据中存在的缺失值、重复记录、异常数据和数据格式不统一等问题，为后续的数据分析和可视化提供可靠的数据基础。

具体目标：

1. 统一字段名称和数据类型，提高数据规范性。
2. 检查并合理处理缺失值，减少无效数据对分析结果的影响。
3. 检查重复记录，避免重复数据影响统计结果。
4. 识别负数量、异常单价和取消订单等特殊业务数据。
5. 根据不同分析需求构建清洗后的数据集。
6. 保存清洗结果，为后续数据分析提供数据支持。

## 二、数据清洗原则

在清洗过程中遵循以下原则：

- **保留原始数据：** 不直接修改原始数据，所有清洗操作均在数据副本上进行。
- **结合业务含义：** 不盲目删除异常数据，需要结合零售交易业务判断数据是否有效。
- **保证可追溯性：** 对重要的数据清洗操作进行记录，方便后续检查和复现。
- **区分分析口径：** 根据不同分析需求构建不同的数据集，避免退货和取消订单影响正常销售统计。

## 三、具体清洗步骤

### 1. 数据格式标准化

主要任务：

- 统一字段名称，去除字段名称前后的空格。
- 将 InvoiceDate 转换为日期时间类型。
- 将 Quantity、Price 和 Customer ID 转换为数值类型。
- 清理 Invoice、StockCode、Description 和 Country 等字符串字段中的多余空格。
- 检查各字段的数据类型是否符合预期。

预期结果：数据字段名称统一，数据类型规范，便于后续处理。

### 2. 缺失值处理

主要任务：

- 使用 `isnull()` 检查各字段的缺失情况。
- 统计每个字段的缺失数量和缺失比例。
- 分析不同字段缺失值的业务含义。
- 根据字段特点决定保留、删除或填充。

处理原则：

- Customer ID 缺失：可以保留记录，但不参与需要客户身份的分析。
- Description 缺失：根据 StockCode 判断是否可以补充，无法确认时保留缺失或在特定分析中排除。
- Invoice、InvoiceDate 等关键字段缺失：根据实际情况判断是否属于无效交易。
- Quantity 和 Price 缺失：优先核查，不随意使用 0 填充。

### 3. 重复值处理

主要任务：

- 使用 `duplicated()` 检查完全重复记录。
- 统计重复记录的数量。
- 分析重复记录产生的原因。
- 检查跨工作表数据是否存在重复。
- 确认属于无效重复记录后再进行删除。

注意事项：

不能仅根据 Invoice 和 StockCode 判断记录是否重复，因为同一订单可能包含相同商品的多条交易记录。

预期结果：识别并处理确认无效的重复记录，避免重复统计。

### 4. 业务异常数据处理

主要任务：

- 检查 Quantity 小于 0 的记录。
- 检查 Price 小于或等于 0 的记录。
- 识别 Invoice 以 C 开头的记录。
- 检查极端交易数量和单价。
- 根据业务规则对特殊记录进行分类和标记。

处理原则：

- **负数量记录：** 可能代表退货或冲销，不直接删除。
- **零单价记录：** 可能代表赠品或特殊交易，需要进一步核查。
- **负单价记录：** 需要单独检查，判断是否属于退款或异常数据。
- **取消订单：** 对 Invoice 以 C 开头的记录进行标记，并结合实际数据验证识别规则。
- **极端值：** 可以使用 IQR（四分位距）方法识别，但不直接删除。

### 5. 构建分析数据集

根据不同的数据分析需求，构建两个数据集。

**（1）清洗后的完整数据集 cleaned_data**

保留具有分析价值的交易记录，包括：

- 正常销售记录
- 退货记录
- 取消订单记录
- 其他具有业务意义的特殊交易记录

主要用于数据质量分析、退货分析和订单行为分析。

**（2）销售分析数据集 sales_data**

从清洗后的数据中筛选符合正常销售统计口径的记录：

- Quantity > 0
- Price > 0
- InvoiceDate 有效

新增销售额字段：

`Sales = Quantity × Price`

主要用于销售额分析、商品销量分析、客户消费分析和国家销售情况分析。

注意：销售分析数据集的筛选条件应根据具体业务分析需求进一步调整。

### 6. 数据质量复检

主要任务：

- 再次检查缺失值。
- 再次检查完全重复记录。
- 检查数值字段是否存在无法转换的数据。
- 检查销售分析数据集是否存在非正数量或非正单价。
- 检查新增 Sales 字段是否存在异常值。
- 对比清洗前后的数据规模。

预期结果：确保清洗后的数据满足后续分析要求。

### 7. 保存清洗结果

将清洗后的数据保存到项目的 `data/processed/` 目录下。

计划输出文件：

| 文件名称 | 文件说明 |
|---|---|
| cleaned_data.csv | 清洗后的完整数据集 |
| sales_data.csv | 用于正常销售分析的数据集 |
| cleaning_report.csv | 数据清洗过程及质量检查报告 |

## 四、预期成果

完成本阶段后，预期得到以下成果：

1. 一份格式规范、具有可追溯性的完整清洗数据集。
2. 一份符合正常销售统计口径的销售分析数据集。
3. 一份记录数据清洗过程和处理结果的质量报告。
4. 为后续销售趋势分析、商品分析、客户分析和可视化工作提供可靠的数据基础。

## 五、清洗流程

原始数据加载
↓
数据格式标准化
↓
缺失值检查与处理
↓
重复值检查与处理
↓
业务异常数据识别与处理
↓
构建分析数据集
↓
数据质量复检
↓
保存清洗结果

---

**本阶段总结：**

数据清洗不仅是删除错误数据，更重要的是结合实际业务理解数据。对于退货、取消订单和特殊价格等记录，应尽可能保留其业务信息，并根据不同的分析目标选择合适的数据口径。

### 加载原数据

In [5]:
df = load_data(RAW_DATA_FILE)
# 备份原始数据
df_raw = df.copy()

print("数据读取成功！")
print("原始数据规模：", df.shape)

csv数据读取成功！
数据规模：1067371行 8列
数据读取成功！
原始数据规模： (1067371, 8)
